### Jupyter Notebook to test code in cells before adding to Python script in .py file

### Data sources:
- populations https://www.michigan-demographics.com/counties_by_population
- geopoints https://public.opendatasoft.com/explore/dataset/us-county-boundaries/export/?flg=en-us&disjunctive.statefp&disjunctive.countyfp&disjunctive.name&disjunctive.namelsad&disjunctive.stusab&disjunctive.state_name&sort=stusab
- shapefile https://public.opendatasoft.com/explore/dataset/us-county-boundaries/export/?flg=en-us&disjunctive.statefp&disjunctive.countyfp&disjunctive.name&disjunctive.namelsad&disjunctive.stusab&disjunctive.state_name&sort=stusab&refine.statefp=26

In [1]:
# Import necessary libraries
import numpy as np
import geopandas as gpd   
import pandas as pd 
from math import pi, pow, sin, cos, asin, sqrt, floor
from pulp import *

#### Precursor 1: Read in the Excel file and GeoJSON file


In [2]:
# Read in the Excel file of Michigan counties
xlsx_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties.xlsx'
michigan_counties = pd.read_excel(xlsx_file_path)

# Read in the GeoJSON file of Michigan county boundaries
geojson_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties.geojson'
michigan_counties_geojson = gpd.read_file(geojson_file_path)

# michigan_counties.head()
# michigan_counties_geojson.head()

#### Precursor 2: Merge the two data sources on country names to have a comprehensive dataset.
- In the 'michigan_counties' dataframe, the county name column is 'county_names', and in the 'michigan_counties_geojson' datafram, the county name column is 'name'.

In [3]:
# Merge the two dataframes on the county name
michigan_counties_merged = michigan_counties.merge(michigan_counties_geojson, left_on='county_names', right_on='name', how='inner')
michigan_counties_merged

,count_id,county_names,latitude,longitude,pop2020,geo_point_2d,statefp,countyfp,countyns,geoid,...,cbsafp,metdivfp,funcstat,aland,awater,intptlat,intptlon,state_name,countyfp_nozero,geometry
0,0,Leelanau,45.151771,-86.038496,22870,"{'lon': -86.0384960523, 'lat': 45.151770859}",26,089,01622987,26089,...,45900,None,A,899241895,5659105307,+45.1461816,-086.0515740,Michigan,89,"POLYGON ((-85.56175 44.95226, -85.56209 44.950..."
1,1,Clinton,42.943652,-84.601517,79748,"{'lon': -84.6015165533, 'lat': 42.9436523662}",26,037,01622961,26037,...,29620,None,A,1467017475,21098128,+42.9504550,-084.5916949,Michigan,37,"POLYGON ((-84.83762 43.03264, -84.83754 43.032..."
2,2,Wexford,44.338367,-85.578414,34196,"{'lon': -85.5784138137, 'lat': 44.3383668115}",26,165,01623023,26165,...,15620,None,A,1463148726,27182043,+44.3313751,-085.5700462,Michigan,165,"POLYGON ((-85.81909 44.42450, -85.81910 44.425..."
3,3,Branch,41.916119,-85.059044,44531,"{'lon': -85.0590443604, 'lat': 41.9161186535}",26,023,01622954,26023,...,17740,None,A,1311605515,34420092,+41.9184551,-085.0668852,Michigan,23,"POLYGON ((-85.29293 41.98482, -85.29293 41.984..."
4,4,Ionia,42.945094,-85.074603,66809,"{'lon': -85.0746031181, 'lat': 42.9450938315}",26,067,01622976,26067,...,24340,None,A,1479710906,22590318,+42.9446503,-085.0737660,Michigan,67,"POLYGON ((-85.07503 43.12021, -85.06470 43.120..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,78,Lapeer,43.090147,-83.221784,88780,"{'lon': -83.2217837238, 'lat': 43.0901474299}",26,087,01622986,26087,...,19820,47664,A,1675675444,41083140,+43.0886327,-083.2243247,Michigan,87,"POLYGON ((-83.45966 43.14338, -83.45966 43.145..."
79,79,Arenac,44.042885,-83.747242,15089,"{'lon': -83.7472418424, 'lat': 44.0428854726}",26,011,01622948,26011,...,None,None,A,940627657,822313309,+44.0368327,-083.7406755,Michigan,11,"POLYGON ((-83.92749 44.16183, -83.92692 44.161..."
80,80,Charlevoix,45.502498,-85.373250,26293,"{'lon': -85.373250216, 'lat': 45.5024976764}",26,029,01622957,26029,...,None,None,A,1078288786,2522931119,+45.5131638,-085.4503920,Michigan,29,"POLYGON ((-85.40467 45.20749, -85.42159 45.207..."
81,81,Alcona,44.683623,-83.129008,10417,"{'lon': -83.1290075152, 'lat': 44.6836233781}",26,001,01622943,26001,...,None,None,A,1747349266,2890404970,+44.6825347,-082.8340784,Michigan,1,"POLYGON ((-83.88742 44.59538, -83.88742 44.595..."


#### Precursor 2: Obtain adjacency information for the countries:
- This will be done by checking which counties share a coundary using the GeoJSON file.

In [4]:
# Determine adjacency between counties based on their geometries
adjacency_matrix = []

# Loop through each county in the GeoJSON file and determine its adjacency to all other counties
for index_1, county_1 in michigan_counties_merged.iterrows(): # Loop through each county in the GeoJSON file
    neighbors = [] # Create an empty list to store the names of neighboring counties
    for index_2, county_2 in michigan_counties_merged.iterrows(): # Loop through each county in the GeoJSON file
        # Check if the two counties are the same to avoid self-comparison
        if county_1['name'] != county_2['name']:
            if county_1['geometry'].touches(county_2['geometry']): # Check if the two counties touch
                neighbors.append(county_2['name']) # If they do, add the name of the second county to the list of neighbors
    adjacency_matrix.append(neighbors) # Add the list of neighbors to the adjacency matrix

# Add adjacency information to the dataframe
michigan_counties_merged['adjacent_counties'] = adjacency_matrix

# Check the adjacency information for the first few counties
michigan_counties_merged[['name', 'adjacent_counties']].head()

,name,adjacent_counties
0,Leelanau,"[Schoolcraft, Antrim, Benzie, Grand Traverse, ..."
1,Clinton,"[Ionia, Shiawassee, Ingham, Montcalm, Eaton, G..."
2,Wexford,"[Missaukee, Osceola, Manistee, Benzie, Lake, K..."
3,Branch,"[Calhoun, St. Joseph, Kalamazoo, Hillsdale]"
4,Ionia,"[Clinton, Barry, Kent, Montcalm, Eaton, Gratiot]"


### Section 1: Calculation Distance Between County Pairs

In [5]:
# Define function to calculate distance between two sets of longitudes / latitudes

# Function to convert degrees to radians
def degrees_to_radians(x):
    return((pi / 180) * x)

# Function to calculate distance between two points on a sphere (in miles) given their longitudes and latitudes
def lon_lat_distance_miles(lon_a, lat_a, lon_b, lat_b):
    """
    Calculates the great-circle distance between two points on a sphere given their longitudes and latitudes.
    
    Parameters:
    lon_a (float): longitude of point A in degrees
    lat_a (float): latitude of point A in degrees
    lon_b (float): longitude of point B in degrees
    lat_b (float): latitude of point B in degrees
    
    Returns:
    float: distance between the two points in miles
    """
    radius_of_earth = 24872 / (2 * pi)
    c = sin((degrees_to_radians(lat_a) - \
    degrees_to_radians(lat_b)) / 2)**2 + \
    cos(degrees_to_radians(lat_a)) * \
    cos(degrees_to_radians(lat_b)) * \
    sin((degrees_to_radians(lon_a) - \
    degrees_to_radians(lon_b))/2)**2
    return(2 * radius_of_earth * (asin(sqrt(c))))

# Function to convert the distance between two points on a sphere (in miles) to meters
def lon_lat_distance_meters (lon_a, lat_a, lon_b, lat_b):
    return(lon_lat_distance_miles(lon_a, lat_a, lon_b, lat_b) * 1609.34)

In [6]:
# Remove population to allow easy joining of long and lat for each county pair
lat_lon = ['county_names', 'latitude', 'longitude']
lat_lon = michigan_counties_merged[lat_lon]

# Create list of county names for pairing        
county_names = michigan_counties['county_names'].to_numpy()

# Create each unique pair
pairs = []

# Loop through each county name and create a pair with each other county name
for i in range(len(county_names)):
    for j in range(i + 1, len(county_names)):
        pairs.append((county_names[i], county_names[j]))

# Create column names for county pairs df
col_names = ['county_1', 'county_2']

# Create df of county pairs                
county_pairs = pd.DataFrame(pairs, columns = col_names)

 # Add first county longitude and latitude
county_pairs = county_pairs.merge(lat_lon, left_on = 'county_1', right_on = 'county_names', how = 'left')
county_pairs.drop('county_names', axis = 1, inplace = True) # Drop county names column
county_pairs = county_pairs.rename(columns={'latitude': 'county_1_lat', 'longitude': 'county_1_long'}) # Rename columns

# Add second county longitude and latitude
county_pairs = county_pairs.merge(lat_lon, left_on = 'county_2', right_on = 'county_names', how = 'left')
county_pairs.drop('county_names', axis = 1, inplace = True) # Drop county names column
county_pairs = county_pairs.rename(columns={'latitude': 'county_2_lat', 'longitude': 'county_2_long'}) # Rename columns

# Add distance between each county pair in miles and meters;
distance_miles = [] # Create empty list to store distance in miles
distance_meters = [] # Create empty list to store distance in meters

# Loop through each county pair and calculate distance in miles and meters
for i in range(len(county_pairs)):
    distance_miles.append(lon_lat_distance_miles(county_pairs.iloc[i, 2], county_pairs.iloc[i, 3], county_pairs.iloc[i, 4], county_pairs.iloc[i, 5]))
    distance_meters.append(lon_lat_distance_meters(county_pairs.iloc[i, 2], county_pairs.iloc[i, 3], county_pairs.iloc[i, 4], county_pairs.iloc[i, 5]))

# Add distance columns to county pairs df
county_pairs['distance_miles'] = distance_miles
county_pairs['distance_meters'] = distance_meters

# Check table
county_pairs

,county_1,county_2,county_1_lat,county_1_long,county_2_lat,county_2_long,distance_miles,distance_meters
0,Leelanau,Clinton,45.151771,-86.038496,42.943652,-84.601517,100.038253,160995.562830
1,Leelanau,Wexford,45.151771,-86.038496,44.338367,-85.578414,32.050066,51579.452482
2,Leelanau,Branch,45.151771,-86.038496,41.916119,-85.059044,69.831366,112382.410744
3,Leelanau,Ionia,45.151771,-86.038496,42.945094,-85.074603,67.621439,108825.886417
4,Leelanau,Mecosta,45.151771,-86.038496,43.640768,-85.324634,49.938224,80367.581755
...,...,...,...,...,...,...,...,...
3398,Arenac,Alcona,44.042885,-83.747242,44.683623,-83.129008,43.010988,69219.303841
3399,Arenac,Livingston,44.042885,-83.747242,42.602917,-83.911528,15.593543,25095.312007
3400,Charlevoix,Alcona,45.502498,-85.373250,44.683623,-83.129008,155.151831,249692.047763
3401,Charlevoix,Livingston,45.502498,-85.373250,42.602917,-83.911528,102.674475,165238.138922


### Stefan Jenss's Method 1:

**Objective Function:**
- Minimize the lack of compactness (a.k.a., the sum of distances between counties that are assigned to the same district).

**Constraints:**
1. Each county should be assigned to expactly one distict.
2. The population in each district should be approximately equal.

**Possible next steps:**
- Add the constraint the counties within a district should be geographically adjacent.

In [19]:
# Create a dictionary of county names and their populations
county_populations = michigan_counties_merged[['name', 'pop2020']].set_index('name').to_dict()['pop2020']

# Number of counties and districts in Michigan
n_counties = 83
n_districts = 14

# Approximate population of each district
average_district_population = sum(county_populations.values()) / n_districts
# Output: Average population = 716722.35

# Initialize the model
model = LpProblem("Michigan_Redistricting", LpMinimize)

# Decision variable: binary variable indicating whether a county is in a district
x = LpVariable.dicts("x", ((i, j) for i in county_names for j in range(n_districts)), cat='Binary')

# Objective Function: Minimize the total distance between counties in the same district
model += lpSum(county_pairs.iloc[i, 7] * x[(county_pairs.iloc[i, 0], j)] for i in range(len(county_pairs)) for j in range(n_districts))

# Constraints: Each county must be assigned to exactly one district
for i in range(len(county_names)):
    model += lpSum(x[(county_names[i], j)] for j in range(n_districts)) == 1

# Constraints: Each district must have a population within 5% of the average population
for j in range(n_districts):
    model += lpSum(county_populations[county_names[i]] * x[(county_names[i], j)] for i in range(len(county_names))) >= 0.95 * average_district_population # Lower bound
    model += lpSum(county_populations[county_names[i]] * x[(county_names[i], j)] for i in range(len(county_names))) <= 1.05 * average_district_population # Upper bound

# Solve the model
model.solve()

# Print the status of the solution
print("Status:", LpStatus[model.status])

# Print the results by county name and district number, along with the total population in each district and the population of each county
for j in range(n_districts):
    print("District", j + 1, ":")
    district_counties = [i for i in range(n_counties) if x[(county_names[i], j)].varValue == 1]
    district_population = sum([county_populations[county_names[i]] for i in district_counties])
    print("Total Population:", district_population)
    for i in range(len(county_names)):
        if x[(county_names[i], j)].varValue == 1:
            print(county_names[i], "Population:", county_populations[county_names[i]])
    print("")




Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/stefanjenss/anaconda3/envs/MSDS460/lib/python3.11/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/t2/xw_g33n96rxfmq14n3dnpb6h0000gn/T/fffb4de1b4714790ab8e86e8896e12ea-pulp.mps timeMode elapsed branch printingOptions all solution /var/folders/t2/xw_g33n96rxfmq14n3dnpb6h0000gn/T/fffb4de1b4714790ab8e86e8896e12ea-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 116 COLUMNS
At line 7075 RHS
At line 7187 BOUNDS
At line 8350 ENDATA
Problem MODEL has 111 rows, 1162 columns and 3486 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 6.64813e+08 - 0.00 seconds
Cgl0000I Cut generators found to be infeasible! (or unbounded)
Pre-processing says infeasible or unbounded
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.05   (Wallclock seconds):       0.06


### Method 2: Implemetation of Forest's Large Country -> District Observation

#### Approach:
1. **Adjust the Dataset**
- Remove the three large counties, Macomb, Oakland, and Wayne from the dataset, since they will manually be assigned to their own districts.
2. **Adjust the number of districts and recalculate the average distict population**
- Since we are removing the three largest counties, we will be to recalculate the average size we want our disticts to be based on the remaining counties.
3. **Modify and Re-run the Model**
- An additional modification I made when re-running this model, what I adjusted the district population constraint to be +/- 20%. This was the only way I was able to find an optimal solution.

In [21]:
### Remove the three large counties

# Identify the large counties in a list
large_counties = ['Wayne', 'Oakland', 'Macomb']

# Create a new dataframe with the large counties removed
michigan_counties_adjusted = michigan_counties_merged[~michigan_counties_merged['name'].isin(large_counties)]

In [23]:
### Ajust the number of counties districts and calculate the new average district population

# Adjust the number of counties and districts in Michigan
n_counties_adjusted = len(michigan_counties_adjusted)
n_districts_adjusted = n_districts - len(large_counties)

# Approximate population of each district
average_district_population_adjusted = michigan_counties_adjusted['pop2020'].sum() / n_districts_adjusted

557585.8181818182

In [27]:
# Filter the county pairs dataframe to only include counties that are not in the large counties list
county_pairs_adjusted = county_pairs[~county_pairs['county_1'].isin(large_counties)]

In [33]:
### Modify and re-run the model

# Create a dictionary of the adjusted county names and their populations
county_populations_adjusted = michigan_counties_adjusted[['name', 'pop2020']].set_index('name').to_dict()['pop2020']
county_names_adjusted = michigan_counties_adjusted['name'].to_numpy()

# Initialize the adjusted model
model_adjusted = LpProblem("Michigan_Redistricting_Adjusted", LpMinimize)

# Decision variable: binary variable indicating whether a county is in a district
x_adjusted = LpVariable.dicts("x_adjusted", ((i, j) for i in county_names_adjusted for j in range(n_districts_adjusted)), cat='Binary')

# Objective Function: Minimize the total distance between counties in the same district
model_adjusted += lpSum(county_pairs_adjusted.iloc[i, 7] * x_adjusted[(county_pairs_adjusted.iloc[i, 0], j)] for i in range(len(county_pairs_adjusted)) for j in range(n_districts_adjusted))

# Constraints: Each county must be assigned to exactly one district
for i in range(len(county_names_adjusted)):
    model_adjusted += lpSum(x_adjusted[(county_names_adjusted[i], j)] for j in range(n_districts_adjusted)) == 1

# Constraints: Each district must have a population within 20% of the average population
for j in range(n_districts_adjusted):
    model_adjusted += lpSum(county_populations_adjusted[county_names_adjusted[i]] * x_adjusted[(county_names_adjusted[i], j)] for i in range(len(county_names_adjusted))) >= 0.80 * average_district_population_adjusted # Lower bound
    model_adjusted += lpSum(county_populations_adjusted[county_names_adjusted[i]] * x_adjusted[(county_names_adjusted[i], j)] for i in range(len(county_names_adjusted))) <= 1.20 * average_district_population_adjusted # Upper bound

# Solve the adjusted model
model_adjusted.solve()

# Print the status of the solution
print("Status:", LpStatus[model_adjusted.status])

# Print the results by county name and district number, along with the total population in each district and the population of each county
for j in range(n_districts_adjusted):
    print("District", j + 1, ":")
    district_counties = [i for i in range(n_counties_adjusted) if x_adjusted[(county_names_adjusted[i], j)].varValue == 1]
    district_population = sum([county_populations_adjusted[county_names_adjusted[i]] for i in district_counties])
    print("Total Population:", district_population)
    for i in range(len(county_names_adjusted)):
        if x_adjusted[(county_names_adjusted[i], j)].varValue == 1:
            print(county_names_adjusted[i], "Population:", county_populations_adjusted[county_names_adjusted[i]])
    print("")

# Manually assign the three large counties to their own districts
for county in large_counties:
    print("District", j + 2, ":")
    print(county, "Population:", county_populations[county])
    j += 1
    print("")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/stefanjenss/anaconda3/envs/MSDS460/lib/python3.11/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/t2/xw_g33n96rxfmq14n3dnpb6h0000gn/T/fbf9be72b8c348d8852d3cb049c1fd1a-pulp.mps timeMode elapsed branch printingOptions all solution /var/folders/t2/xw_g33n96rxfmq14n3dnpb6h0000gn/T/fbf9be72b8c348d8852d3cb049c1fd1a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 107 COLUMNS
At line 5377 RHS
At line 5480 BOUNDS
At line 6361 ENDATA
Problem MODEL has 102 rows, 880 columns and 2640 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 6.50032e+08 - 0.00 seconds
Cgl0005I 80 SOS with 880 members
Cgl0004I processed model has 91 rows, 880 columns (880 integer (880 of which binary)) and 1760 elements
Cbc0038I Initial state - 8 integers unsatisfied sum - 2.80353
Cbc0038I Pass   1: suminf.    1.55